# Host expression source fidelity and GTEx version comparison


GT-HOST-TX-01. Compare only the same source quantities/selected fields and preserve original gene rows. Read verified acquisition receipts and fixed reviewed UniProt mapping. Replay native loaders through explicitly recorded temporary symlinks; installed data and archive inputs are never overwritten. GTEx v11 remains a candidate; rank/coverage changes do not prove improved biological accuracy. CAGE and RNA-seq remain separate.

In [1]:
import json, hashlib, gzip, tempfile
import pandas as pd, numpy as np
from pathlib import Path
from starplast import host as H, organisms as O
from scripts.recover_source_files import download
root=Path('/media/carruthers/mnt3/claude/toxoplasma_projects/datasets')
base=Path('results/host_expression_source_review_2026_10_08')
out=base/'comparison_v3'
records={r['source_id']:r for r in json.loads((base/'sources.json').read_text())}
def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1<<20),b''): digest.update(block)
    return digest.hexdigest()
inputs=[base/'sources.json',base/'fantom_resources.json',Path('starplast/host.py'),Path('scripts/recover_source_files.py')]
for species,names in H.UNIPROT_IDMAP.items():
    inputs.extend(root/H.UNIPROT_ROOT/name for name in names)
for record in records.values():
    if 'file' in record: assert sha(record['file']['path'])==record['file']['sha256']
    if 'verified_files' in record:
        for file in record['verified_files']: assert sha(file['absolute_path'])==file['sha256']
candidate=records['host_gtex_transcriptome']['candidate_v11']
assert sha(candidate['file']['path'])==candidate['file']['sha256']
paths={'v10':Path(records['host_gtex_transcriptome']['file']['path']),'v11':Path(candidate['file']['path'])}
frames={}
for version,path in paths.items():
    inputs.append(path)
    with gzip.open(path,'rt') as stream:
        assert stream.readline().strip()=='#1.2'
        shape=tuple(map(int,stream.readline().strip().split('\t')))
        frame=pd.read_csv(stream,sep='\t')
    assert shape==(len(frame),len(frame.columns)-2)
    selected=frame[['Name','Description',*H.GTEX_TISSUES]].copy()
    selected['native_core_gene']=selected.Name.str.split('.').str[0]
    selected['ensembl_gene']=selected.Name.str.replace(r'\.\d+(?=_PAR_Y$|$)','',regex=True)
    print(version,'native core duplicate rows',int(selected.native_core_gene.duplicated(keep=False).sum()))
    assert selected.ensembl_gene.is_unique
    selected.to_parquet(out/(version+'_selected_gene_rows.parquet'),index=False)
    frames[version]=selected.set_index('ensembl_gene')
print('Original gene universes', {k:len(v) for k,v in frames.items()})

v10 native core duplicate rows 90
v11 native core duplicate rows 0
Original gene universes {'v10': 59033, 'v11': 74628}


In [2]:
resources=json.loads((base/'fantom_resources.json').read_text())
matching=[r for r in resources if r['description']=='Expression values across all genes (tpms)' and r['url'].endswith('/tpmss.tsv')]
assert len(matching)==1 and matching[0]['url'].startswith('experiments-content/E-MTAB-3579/')
url='https://www.ebi.ac.uk/gxa/'+matching[0]['url']
original=root/'host_expression_review_inputs/FANTOM5_E-MTAB-3579/tpmss.tsv'
fresh={'url':url,'status':'unavailable_continue'}
try:
    if not original.exists(): download(url,original,limit=32<<20)
    old=Path(next(f['absolute_path'] for f in records['host_mouse_tissue_transcriptome']['verified_files'] if f['name']=='E-MTAB-3579-mouse-tissue-tpm.tsv'))
    inputs.extend((original,old))
    before=pd.read_csv(old,sep='\t',comment='#')
    after=pd.read_csv(original,sep='\t',comment='#')
    pd.testing.assert_frame_equal(before,after)
    fresh.update(status='all_primary_table_values_exact_to_historical_local_table',sha256=sha(original),historical_sha256=sha(old),rows=len(after),columns=len(after.columns))
except Exception as error:
    fresh['gap']=type(error).__name__+': '+str(error)
(out/'fantom_primary_equivalence.json').write_text(json.dumps(fresh,indent=2)+'\n')
print(fresh)

{'url': 'https://www.ebi.ac.uk/gxa/experiments-content/E-MTAB-3579/resources/ExperimentDownloadSupplier.RnaSeqBaseline/tpmss.tsv', 'status': 'all_primary_table_values_exact_to_historical_local_table', 'sha256': 'ee5debd905add8e7a3dddcc2bdd05d1d0957deef1ee85e607b737d5f065f3792', 'historical_sha256': '69d84aafc882fcadf723cb4538937d887f01cdb9804f382882ce87b9ab8a9928', 'rows': 18835, 'columns': 172}


In [3]:
summaries={}
native={}
with tempfile.TemporaryDirectory(prefix='starplast-host-tx-review-') as temp:
    staging=Path(temp)
    reference=staging/H.UNIPROT_ROOT
    reference.parent.mkdir(parents=True)
    reference.symlink_to(root/H.UNIPROT_ROOT,target_is_directory=True)
    folder=staging/'host'/H.GTEX[0]/H.GTEX[1]
    folder.mkdir(parents=True)
    for version,path in paths.items():
        link=folder/H.GTEX[2]
        link.symlink_to(path)
        native[version]=H.gtex_transcriptome(str(staging)).set_index('host_id')
        link.unlink()
    mouse_path=Path(next(f['absolute_path'] for f in records['host_mouse_tissue_transcriptome']['verified_files'] if f['name']=='E-MTAB-3579-mouse-tissue-tpm.tsv'))
    mouse_folder=staging/'host'/H.FANTOM5_MOUSE[0]/H.FANTOM5_MOUSE[1]
    mouse_folder.mkdir(parents=True)
    (mouse_folder/H.FANTOM5_MOUSE[2]).symlink_to(mouse_path)
    native['fantom']=H.mouse_tissue_transcriptome(str(staging)).set_index('host_id')
for key,organism,columns in [('v10',O.HUMAN,list(H.GTEX_TISSUES.values())),('fantom',O.MOUSE,['brain_tpm','skeletal_muscle_tpm'])]:
    installed_path=Path('starplast/data')/O.HOST_TABLES[organism]
    inputs.append(installed_path)
    installed=pd.read_parquet(installed_path).set_index('host_id')
    summaries[key]={'native_mapped_rows':len(native[key]),'fields':{}}
    for column in columns:
        observed=installed[column].dropna()
        projected=native[key][column].reindex(installed.index)
        unsupported=observed.index.difference(native[key].index).tolist()
        missing=int((installed[column].notna() & projected.isna()).sum())
        extra=int((installed[column].isna() & projected.notna()).sum())
        overlap=installed[column].notna() & projected.notna()
        differences=int((installed.loc[overlap,column].astype(float).to_numpy()!=projected[overlap].astype(float).to_numpy()).sum())
        summaries[key]['fields'][column]={'installed_observed':len(observed),'missing_reproductions':missing,'extra_mapped_values_on_installed_universe':extra,'exact_differences':differences,'unresolved_installed_IDs':unsupported}
        replay=pd.DataFrame({'host_id':installed.index,'installed_value':installed[column].to_numpy(),'replayed_value':projected.to_numpy()})
        replay.to_parquet(out/(key+'_'+column+'_replay_rows.parquet'),index=False)
    native[key].reset_index().to_parquet(out/(key+'_native_projection.parquet'),index=False)
native['v11'].reset_index().to_parquet(out/'v11_candidate_native_projection.parquet',index=False)
print(json.dumps(summaries,indent=2))

uniprot index (human): 21,844 Ensembl genes, 20,190 symbols; 199 Ensembl ids dropped for naming two reviewed proteins
gtex: 19,088 human genes over 2 tissues (39,808 of 59,033 rows had no reviewed UniProt accession and are left out)
uniprot index (human): 21,844 Ensembl genes, 20,190 symbols; 199 Ensembl ids dropped for naming two reviewed proteins
gtex: 19,099 human genes over 2 tissues (55,402 of 74,628 rows had no reviewed UniProt accession and are left out)
uniprot index (mouse): 15,656 Ensembl genes, 16,955 symbols; 55 Ensembl ids dropped for naming two reviewed proteins
mouse tissue: 13,851 genes; brain averaged over 4 adult regions (4,939 of 18,835 rows had no reviewed accession)
{
  "v10": {
    "native_mapped_rows": 19088,
    "fields": {
      "fibroblast_tpm": {
        "installed_observed": 19087,
        "missing_reproductions": 3,
        "extra_mapped_values_on_installed_universe": 2,
        "exact_differences": 0,
        "unresolved_installed_IDs": [
          "P59510

In [4]:
common=frames['v10'].index.intersection(frames['v11'].index)
comparison={'v10_genes':len(frames['v10']),'v11_genes':len(frames['v11']),'shared_genes':len(common),'v10_only_genes':len(frames['v10'].index.difference(frames['v11'].index)),'v11_only_genes':len(frames['v11'].index.difference(frames['v10'].index)),'v10_native_mapped_proteins':len(native['v10']),'v11_native_mapped_proteins':len(native['v11']),'shared_mapped_proteins':len(native['v10'].index.intersection(native['v11'].index)),'fields':{},'decision':'candidate_only; annotation and sample/method lineage plus affected validation pending'}
for field in H.GTEX_TISSUES:
    old=frames['v10'].loc[common,field].astype(float)
    new=frames['v11'].loc[common,field].astype(float)
    finite=np.isfinite(old)&np.isfinite(new)
    comparison['fields'][field]={'common_finite_gene_rows':int(finite.sum()),'exact_equal':int((old[finite]==new[finite]).sum()),'changed':int((old[finite]!=new[finite]).sum()),'spearman_rank_correlation':float(old[finite].corr(new[finite],method='spearman')),'median_absolute_TPM_difference':float((old[finite]-new[finite]).abs().median())}
(out/'comparison.json').write_text(json.dumps({'installed_reproduction':summaries,'gtex_version_comparison':comparison},indent=2,allow_nan=False)+'\n')
(out/'manifest.json').write_text(json.dumps({'input_sha256':{str(p.resolve()):sha(p) for p in inputs},'outputs':{p.name:sha(p) for p in out.iterdir() if p.is_file()}},indent=2)+'\n')
print(json.dumps(comparison,indent=2))

{
  "v10_genes": 59033,
  "v11_genes": 74628,
  "shared_genes": 56474,
  "v10_only_genes": 2559,
  "v11_only_genes": 18154,
  "v10_native_mapped_proteins": 19088,
  "v11_native_mapped_proteins": 19099,
  "shared_mapped_proteins": 19082,
  "fields": {
    "Cells_Cultured_fibroblasts": {
      "common_finite_gene_rows": 56474,
      "exact_equal": 29609,
      "changed": 26865,
      "spearman_rank_correlation": 0.980912572935858,
      "median_absolute_TPM_difference": 0.0
    },
    "Liver_Hepatocyte": {
      "common_finite_gene_rows": 56474,
      "exact_equal": 30316,
      "changed": 26158,
      "spearman_rank_correlation": 0.9801465208419708,
      "median_absolute_TPM_difference": 0.0
    }
  },
  "decision": "candidate_only; annotation and sample/method lineage plus affected validation pending"
}


Initial comparison stopped before writing source rows because the native split-at-dot identifier transform collapses PAR_Y suffixes and creates duplicate gene keys. Preserve that diagnostic code in comparison/initial_comparison_code.py. This revision retains original names and strips only version digits while preserving PAR_Y; the native projection is replayed unchanged for fidelity, not endorsed as gene-level truth. No installed values changed. The second comparison correctly refused a non-exact installed mapping association; its code and partial outputs remain in comparison_v2. This revision records every unsupported key and numerical difference instead of stopping the other source review. Native source replay is fidelity evidence, not measurement accuracy. New/removed genes and changed gene models require an explicit mapping/quality decision and selection-policy record; more rows alone do not identify biological comprehensiveness. Same GTEx family releases share original publications/donors; do not give v11 a new citation-age clock. Missing Atlas values remain missing rather than biological negatives. Host gene-space and inference-pack action items remain open.